# Simple: Categorical Field Analyzer with PAMOLA.CORE

**Goal**: Learn categorical field analysis basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Analyze categorical field distributions using execute()
- Detect anomalies (typos, rare values, single characters)
- Understand distribution characteristics and diversity metrics

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/
        └── 01_categorical_analyzer_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.categorical import (
        CategoricalOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load data from examples/data_examples/sample.csv
- Auto-creates sample data if file doesn't exist
- Review dataset structure and statistics

**What you'll see:**
- File path confirmation or creation message
- Dataset summary (200 records, 7 columns)
- First 5 records preview
- Column statistics (types, unique values, ranges)
- Sample includes categorical fields: product_category, review_status, customer_region

**Note:** Sample data includes intentional anomalies (typos, single characters) for demonstration

In [ ]:
# Define path to sample data (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

# Check if file exists
if not data_path.exists():
    print(f"⚠️  File not found: {data_path}")
    print("Creating sample data...\n")
    
    # Create directory
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample product review data with various categorical fields
    np.random.seed(42)
    
    # Define categories with some intentional anomalies
    statuses = ['Approved', 'Pending', 'Rejected', 'Approved', 'Pending'] * 40
    statuses += ['Approvd', 'Aproved', 'P', 'R', 'N']  # Add typos and single chars
    
    categories = ['Electronics', 'Clothing', 'Books', 'Home & Garden', 'Sports'] * 38
    categories += ['Eletronics', 'Electrnics', 'Books', 'B', '123', 'Sports', 'Clothng', 'Clth', 'Books', 'Home & Grdn']
    
    regions = ['North America', 'Europe', 'Asia', 'South America', 'Africa'] * 38
    regions += ['NA', 'Eurpe', 'Aia', 'South Amrica', 'Afric', 'E', 'A', 'N', 'Europe', 'Asia']
    
    sample_data = pd.DataFrame({
        'review_id': [f'R{i:04d}' for i in range(1, 201)],
        'product_category': categories,
        'review_status': statuses,
        'customer_region': regions,
        'rating': np.random.randint(1, 6, 200),
        'verified_purchase': np.random.choice(['Yes', 'No'], 200, p=[0.7, 0.3]),
        'review_length': np.random.randint(10, 500, 200)
    })
    
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created at: {data_path}\n")

# Load data using pandas
df = read_csv(data_path)

print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE field_name**: Edit `create_config_kwargs()` function
   - Change `"field_name": "company_current"` to YOUR column name
   - Available fields: product_category, review_status, customer_region, verified_purchase
2. Run to validate field and setup environment

**What you'll see:**
- Field validation (✓ Field exists, unique values, sample values, distribution)
- Task directory created (✓)
- TaskReporter initialized (✓)
- Config kwargs confirmed (✓)
- DataSource created (✓)
- Progress tracker ready (✓)

**Note:** Field must exist in dataset and be categorical/string type. Default uses 'company_current' which doesn't exist - you MUST customize this!

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "product_category"  # Customize this!
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "company_current"  # Field to analyze - CUSTOMIZE THIS!
    }

# Get config
kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists in dataset
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

print(f"✓ Field to analyze: '{field_name}'")
print(f"  Unique values: {df[field_name].nunique()}")
print(f"  Sample values: {list(df[field_name].unique()[:5])}")
print(f"  Distribution:")
print(df[field_name].value_counts().head())

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="categorical_001",
    task_type="categorical_analysis",
    description=f"Categorical analysis of '{field_name}' field.",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config kwargs (without field_name for execute)
execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Analyzing {field_name} field",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review operation configuration
- Run to execute categorical analysis
- Monitor progress bar (6 processing steps)

**What you'll see:**
- Configuration summary (field, top_n, settings confirmed)
- Progress bar: validate → analyze → calculate metrics → visualize → save
- Completion message with status and artifact count (4-6 files expected)
- Status = "success" (verify no errors)

**Key parameters:**
- `field_name`: Field to analyze (from config)
- `top_n=15`: Number of top values to show
- `analyze_anomalies=True`: Detect typos and rare values
- `generate_visualization=True`: Create distribution charts
- `save_output=True`: Save all results to files

In [ ]:
# Create operation with production-style configuration
operation = CategoricalOperation(
    # Core parameters
    field_name=field_name,               # From config
    top_n=15,                           # Top values to show
    min_frequency=1,                    # Min frequency for dictionary
    
    # Analysis configuration
    profile_type='categorical',         # Profile type
    analyze_anomalies=True,             # Detect typos/anomalies
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,            # Disable for small data
    parallel_processes=1,
    use_dask=False,
    chunk_size=10000,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,        # Create visualization artifacts
    save_output=True,                   # Save results to files
    force_recalculation=False           # Use cache when available
)

print("✓ Operation configured")
print(f"  Field:            {operation.field_name}")
print(f"  Top N:            {operation.top_n}")
print(f"  Min frequency:    {operation.min_frequency}")
print(f"  Analyze anomalies: {operation.analyze_anomalies}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing categorical analysis...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load and display analysis results
- Review distribution metrics and anomalies
- Check for data quality issues

**What you'll see:**
- Generated artifacts list (statistics, dictionary, anomalies, visualizations)
- Basic statistics (total records, nulls, unique values)
- Diversity metrics (entropy, cardinality ratio, distribution type)
- Top values with frequencies and percentages
- Detected anomalies (typos, single chars, numeric strings)
- Summary metrics from operation result

**Note:** Higher entropy (>2.0) indicates diverse distribution. Check anomalies for data quality issues.

**Generated artifacts:**
- Statistics JSON in output/
- Value dictionary CSV in dictionaries/
- Anomalies CSV in dictionaries/
- Visualizations in visualizations/

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load statistics file
stats_files = list(task_dir.glob('output/*_stats_*.json'))
if stats_files:
    stats_file = stats_files[0]
    
    with open(stats_file, 'r') as f:
        stats_data = json.load(f)
    
    print("\n" + "=" * 80)
    print("📊 ANALYSIS RESULTS")
    print("=" * 80)
    
    # Display basic statistics
    print("\n🔍 Basic Statistics:")
    print("-" * 60)
    print(f"  Total records:      {stats_data.get('total_records', 0):,}")
    print(f"  Non-null values:    {stats_data.get('non_null_values', 0):,}")
    print(f"  Null values:        {stats_data.get('null_values', 0):,} ({stats_data.get('null_percent', 0):.2f}%)")
    print(f"  Unique values:      {stats_data.get('unique_values', 0)}")
    
    # Display diversity metrics
    print("\n📈 Diversity Metrics:")
    print("-" * 60)
    print(f"  Entropy:            {stats_data.get('entropy', 0):.4f}")
    print(f"  Cardinality ratio:  {stats_data.get('cardinality_ratio', 0):.4f}")
    if 'distribution_type' in stats_data:
        print(f"  Distribution type:  {stats_data.get('distribution_type', 'unknown')}")
    if 'concentration' in stats_data:
        print(f"  Concentration:      {stats_data.get('concentration', 0):.4f}")
    if 'skewness' in stats_data:
        print(f"  Skewness:           {stats_data.get('skewness', 0):.4f}")
    
    # Display top values
    if 'top_values' in stats_data and stats_data['top_values']:
        print("\n📊 Top Values Distribution:")
        print("-" * 60)
        top_vals = stats_data['top_values']
        total = stats_data.get('non_null_values', 1)
        
        for value, count in list(top_vals.items())[:10]:
            pct = (count / total) * 100
            bar = '█' * int(pct / 2)
            print(f"  {value:30s} {bar:25s} {count:4d} ({pct:5.1f}%)")
        
        print(f"\n  Coverage by top values: {stats_data.get('percent_covered_by_top', 0):.2f}%")
    
    # Display anomalies
    if 'anomalies' in stats_data and stats_data['anomalies']:
        print("\n⚠️  DETECTED ANOMALIES:")
        print("=" * 60)
        anomalies = stats_data['anomalies']
        
        if 'potential_typos' in anomalies and anomalies['potential_typos']:
            print("\n  Potential Typos:")
            print("  " + "-" * 56)
            for value, info in list(anomalies['potential_typos'].items())[:5]:
                print(f"    '{value}' (count: {info['count']}) → might be '{info['similar_to']}' (count: {info['similar_count']})")
        
        if 'single_char_values' in anomalies and anomalies['single_char_values']:
            print("\n  Single Character Values:")
            print("  " + "-" * 56)
            for value, count in list(anomalies['single_char_values'].items())[:5]:
                print(f"    '{value}': {count} occurrences")
        
        if 'numeric_like_strings' in anomalies and anomalies['numeric_like_strings']:
            print("\n  Numeric-like Strings:")
            print("  " + "-" * 56)
            for value, count in list(anomalies['numeric_like_strings'].items())[:5]:
                print(f"    '{value}': {count} occurrences")
    
    # Display result metrics
    if result.metrics:
        print("\n" + "=" * 80)
        print("📊 SUMMARY METRICS")
        print("=" * 80)
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
            else:
                print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Higher entropy and more unique values = More diverse distribution!")
else:
    print("⚠️  No statistics file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files in directory structure
- Navigate to task_dir for manual inspection
- Verify file counts and sizes

**What you'll see:**
- Directory structure with file counts per folder
- File names with sizes (KB) for up to 5 files per folder
- Full path to task directory for external access
- Total artifacts confirmation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Statistics JSON
├── dictionaries/     # Value dictionary and anomalies CSV
├── visualizations/   # PNG charts
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

# List all subdirectories and files
for subdir in ['output', 'dictionaries', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:  # Show first 5 files
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")
        if len(files) > 5:
            print(f"   ... and {len(files) - 5} more files")
        print()

print("=" * 80)
print("\n✅ All artifacts saved successfully!")
print(f"\n💡 TIP: Navigate to {task_dir.absolute()} to inspect files manually")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Operation performance metrics (if available)
2. **Statistics JSON**: Detailed field analysis with distributions
3. **Value Dictionary**: Frequency table of all values
4. **Anomalies**: Detected typos, single characters, numeric strings
5. **Visualizations**: Distribution charts

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types files
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            # Use only filtered files
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            # Fallback: nothing left after filtering → use all
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]
        
        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metadata and metrics
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            # Display metadata
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")
            if 'operation' in metadata:
                op = metadata['operation']
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")
            
            # Display key metrics
            print("\n📊 KEY METRICS:")
            metric_keys = [
                ('total_records', 'Total Records'),
                ('null_count', 'Null Count'),
                ('null_percent', 'Null Percent'),
                ('unique_values', 'Unique Values'),
                ('entropy', 'Entropy'),
                ('cardinality_ratio', 'Cardinality Ratio'),
                ('distribution_type', 'Distribution Type'),
                ('anomalies_count', 'Anomalies Count')
            ]
            
            for key, label in metric_keys:
                if key in metrics:
                    value = metrics[key]
                    if isinstance(value, float):
                        print(f"   {label}: {value:.4f}")
                    else:
                        print(f"   {label}: {value}")
        
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

print("\n\n2️⃣ STATISTICS (Detailed Analysis):")
print("-" * 80)

output_dir = task_dir / "output"

if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")

else:
    stats_files = list(output_dir.glob("*_stats_*.json"))

    if not stats_files:
        print("⚠️  No statistics files found")

    else:
        # Pick newest
        stats_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_stats_file = stats_files[0]

        mtime = datetime.fromtimestamp(latest_stats_file.stat().st_mtime)
        print(f"✓ Found {len(stats_files)} statistics file(s)")
        print(f"📄 Loading LATEST: {latest_stats_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_stats_file.stat().st_size / 1024:.1f} KB")
        print("=" * 80)

        try:
            with open(latest_stats_file, "r", encoding="utf-8") as f:
                stats_data = json.load(f)

            # ─────────────────────────────────────────────
            # BASIC FIELD INFO
            # ─────────────────────────────────────────────
            print("\n📊 FIELD OVERVIEW:")
            print("-" * 60)

            field_name = stats_data.get("field_name", "N/A")
            total_records = stats_data.get("total_records", 0)
            non_null = stats_data.get("non_null_values", 0)
            nulls = stats_data.get("null_values", 0)
            null_pct = stats_data.get("null_percent", 0)
            unique_vals = stats_data.get("unique_values", 0)

            print(f"  Field Name       : {field_name}")
            print(f"  Total Records    : {total_records:,}")
            print(f"  Non-null Values  : {non_null:,}")
            print(f"  Null Values      : {nulls:,} ({null_pct:.2f}%)")
            print(f"  Unique Values    : {unique_vals:,}")

            # ─────────────────────────────────────────────
            # DISTRIBUTION CHARACTERISTICS
            # ─────────────────────────────────────────────
            print("\n📈 DISTRIBUTION CHARACTERISTICS:")
            print("-" * 60)

            if "entropy" in stats_data:
                print(f"  Entropy             : {stats_data['entropy']:.6f}")
            if "cardinality_ratio" in stats_data:
                print(f"  Cardinality Ratio   : {stats_data['cardinality_ratio']:.6f}")
            if "distribution_type" in stats_data:
                print(f"  Distribution Type   : {stats_data['distribution_type']}")
            if "skewness" in stats_data:
                print(f"  Skewness            : {stats_data['skewness']:.6f}")
            if "concentration" in stats_data:
                print(f"  Concentration       : {stats_data['concentration']:.6f}")

            # ─────────────────────────────────────────────
            # TOP VALUES
            # ─────────────────────────────────────────────
            if "top_values" in stats_data and stats_data["top_values"]:
                print("\n📊 TOP VALUES:")
                print("-" * 70)
                print(f"{'Value':<40} {'Count':>8} {'Percent':>10}")
                print("-" * 70)

                total_non_null = non_null if non_null > 0 else 1

                for value, count in list(stats_data["top_values"].items())[:15]:
                    pct = (count / total_non_null) * 100
                    value_str = str(value)[:37] + "..." if len(str(value)) > 40 else str(value)
                    print(f"{value_str:<40} {count:>8} {pct:>9.2f}%")

                if "percent_covered_by_top" in stats_data:
                    print("-" * 70)
                    print(f"Coverage by top values: {stats_data['percent_covered_by_top']:.2f}%")

            else:
                print("\n📊 TOP VALUES: N/A")

            # ─────────────────────────────────────────────
            # VALUE DICTIONARY (OPTIONAL, DEEP DETAIL)
            # ─────────────────────────────────────────────
            if "value_dictionary" in stats_data:
                vd = stats_data["value_dictionary"]

                print("\n📚 VALUE DICTIONARY SUMMARY:")
                print("-" * 70)

                print(f"  Total Unique Values : {vd.get('total_unique_values', 'N/A')}")
                print(f"  Min Frequency       : {vd.get('min_frequency', 'N/A')}")
                print(f"  Max Frequency       : {vd.get('max_frequency', 'N/A')}")

                if "dictionary_data" in vd and vd["dictionary_data"]:
                    print("\n  Sample Values:")
                    print(f"  {'Value':<40} {'Freq':>8} {'Percent':>10}")
                    print("  " + "-" * 60)

                    for row in vd["dictionary_data"][:10]:
                        val = row.get(field_name, "N/A")
                        freq = row.get("frequency", 0)
                        pct = row.get("percent", 0)
                        val_str = str(val)[:37] + "..." if len(str(val)) > 40 else str(val)
                        print(f"  {val_str:<40} {freq:>8} {pct:>9.2f}%")

            print("\n✅ Statistics rendering completed successfully.")

        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading statistics: {e}")

# 3. VALUE DICTIONARY (NEWEST)
print("\n\n3️⃣ VALUE DICTIONARY:")
print("-" * 80)

dict_dir = task_dir / 'dictionaries'
if not dict_dir.exists():
    print(f"⚠️  Dictionaries directory not found: {dict_dir}")
else:
    dict_files = list(dict_dir.glob('*_dictionary_*.csv'))
    
    if not dict_files:
        print("⚠️  No dictionary files found")
    else:
        # Sort by modification time (newest first)
        dict_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_dict_file = dict_files[0]
        
        # Show file info
        mtime = datetime.fromtimestamp(latest_dict_file.stat().st_mtime)
        print(f"✓ Found {len(dict_files)} dictionary file(s)")
        print(f"📄 Loading LATEST: {latest_dict_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_dict_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            dict_df = pd.read_csv(latest_dict_file)
            
            print(f"📊 Shape: {dict_df.shape[0]} rows × {dict_df.shape[1]} columns")
            print(f"\n📋 Value Frequency Table (Top 20):\n")
            print(dict_df.head(20).to_string(index=False))
            
            if len(dict_df) > 20:
                print(f"\n... and {len(dict_df) - 20} more values")
        
        except Exception as e:
            print(f"❌ Error reading dictionary: {e}")

# 4. ANOMALIES (NEWEST)
print("\n\n4️⃣ DETECTED ANOMALIES:")
print("-" * 80)

anomaly_files = list(dict_dir.glob('*_anomalies_*.csv'))
if not anomaly_files:
    print("   ℹ️  No anomalies file found (no anomalies detected or anomaly detection disabled)")
else:
    # Sort by modification time (newest first)
    anomaly_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    latest_anomaly_file = anomaly_files[0]
    
    mtime = datetime.fromtimestamp(latest_anomaly_file.stat().st_mtime)
    print(f"✓ Found {len(anomaly_files)} anomaly file(s)")
    print(f"📄 Loading LATEST: {latest_anomaly_file.name}")
    print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   Size: {latest_anomaly_file.stat().st_size / 1024:.1f} KB\n")
    
    try:
        anomaly_df = pd.read_csv(latest_anomaly_file)
        
        print(f"📊 Total Anomalies: {len(anomaly_df)}\n")
        
        # Group by anomaly type
        if 'anomaly_type' in anomaly_df.columns:
            print("⚠️  ANOMALIES BY TYPE:\n")
            
            for atype in anomaly_df['anomaly_type'].unique():
                subset = anomaly_df[anomaly_df['anomaly_type'] == atype]
                print(f"  {atype.replace('_', ' ').title()} ({len(subset)} found):")
                print("  " + "-" * 60)
                
                for _, row in subset.head(5).iterrows():
                    if atype == 'potential_typo' and row.get('similar_to'):
                        print(f"    • '{row['value']}' (freq: {row['frequency']}) → "
                              f"might be '{row['similar_to']}' (freq: {row['similar_count']})")
                    else:
                        print(f"    • '{row['value']}': {row['frequency']} occurrences")
                
                if len(subset) > 5:
                    print(f"    ... and {len(subset) - 5} more")
                print()
        else:
            print(anomaly_df.to_string(index=False))
    
    except Exception as e:
        print(f"❌ Error reading anomalies: {e}")

# 5. VISUALIZATIONS (NEWEST BATCH)
print("\n\n5️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                viz_name = viz_file.stem.replace('_', ' ').replace('distribution', 'Distribution').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**

✅ **Load data**: Read CSV from examples/data_examples/  
✅ **Setup environment**: TaskReporter, DataSource, ProgressTracker  
✅ **Configure operation**: Full production-style parameters with field config  
✅ **Execute**: Use operation.execute() with explicit configs  
✅ **Analyze results**: Review distributions, anomalies, and diversity metrics  

**Key takeaways:**
- Automatic frequency distribution analysis
- Entropy measures diversity (higher = more diverse)
- Cardinality ratio = unique values / total values
- Anomaly detection finds typos, single characters, numeric strings
- Distribution types: single_value, highly_concentrated, well_distributed
- Value dictionary includes all values meeting minimum frequency
- All analysis saved in structured directories

---

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_categorical_analyzer_advanced.ipynb`** includes:
- Multi-field batch analysis
- Custom anomaly detection rules
- Advanced distribution metrics
- Large dataset processing with Dask
- Performance optimization techniques
- Integration with data quality pipelines

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 🎓 [Data Quality Guide](../../docs/data_quality.md)
- 📋 [Categorical Analysis Best Practices](../../docs/categorical_analysis.md)